# Fabula News: generador de video gratuito

Convierte **una foto del personaje + el audio del guion** en un video hablado, usando SadTalker en la GPU gratuita de Colab.

**Cómo usarlo cada vez que vayas a producir:**
1. Menú *Entorno de ejecución* → *Cambiar tipo de entorno* → GPU (T4).
2. *Entorno de ejecución* → *Ejecutar todo*. La primera vez tarda ~5 min (descarga modelos).
3. Al final aparece un link `https://xxxx.gradio.live`. Cópialo y pégalo en la app de Fabula News → *Configurar video*.
4. Deja esta pestaña abierta mientras generas videos. El link dura hasta 72 h o hasta que cierres Colab.

In [ ]:
#@title 1. Instalar SadTalker y dependencias (una sola vez por sesión)
import os, subprocess, sys
if not os.path.exists('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker.git /content/SadTalker
%cd /content/SadTalker
!pip install -q -r requirements.txt
!pip install -q gradio==4.44.1 imageio-ffmpeg
!apt-get -qq install -y ffmpeg > /dev/null
print('Dependencias listas')

In [ ]:
#@title 2. Descargar modelos (~2 GB, una sola vez por sesión)
%cd /content/SadTalker
if not os.path.exists('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    !bash scripts/download_models.sh
print('Modelos listos')

In [ ]:
#@title 3. Levantar el servidor y obtener el link para la app
%cd /content/SadTalker
import gradio as gr, subprocess, glob, os, shutil, time

def generar(imagen, audio):
    """Recibe ruta de imagen y ruta de audio, devuelve ruta del mp4."""
    if imagen is None or audio is None:
        raise gr.Error('Faltan la imagen o el audio.')
    out_dir = f'/content/out_{int(time.time())}'
    os.makedirs(out_dir, exist_ok=True)
    cmd = [
        sys.executable, 'inference.py',
        '--driven_audio', audio,
        '--source_image', imagen,
        '--result_dir', out_dir,
        '--still',            # cabeza estable, estilo presentadora
        '--preprocess', 'full',
        '--enhancer', 'gfpgan',  # mejora la cara; quita esta línea si tarda demasiado
        '--size', '256',
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise gr.Error(proc.stderr[-1500:])
    videos = glob.glob(f'{out_dir}/**/*.mp4', recursive=True)
    if not videos:
        raise gr.Error('SadTalker no generó video. Revisa que la cara se vea de frente.')
    final = f'{out_dir}/fabula.mp4'
    shutil.copy(videos[-1], final)
    return final

demo = gr.Interface(
    fn=generar,
    inputs=[gr.Image(type='filepath', label='Imagen del personaje'),
            gr.Audio(type='filepath', label='Audio del guion')],
    outputs=gr.Video(label='Video'),
    title='Fabula News video server',
    api_name='generar',
)
demo.queue().launch(share=True)

### Si algo falla
- **"No se pudo asignar GPU"**: Colab gratis a veces no tiene T4 disponible; espera un rato y vuelve a intentar.
- **Muy lento**: en la celda 3 quita la línea `'--enhancer', 'gfpgan',`. Un guion de 45 s tarda entre 3 y 8 min en T4.
- **Error de versión de Python o torch**: SadTalker es un proyecto de 2023; si Colab actualizó Python, en la celda 1 agrega antes de `pip install`: `!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121`.
- Para mejor calidad en el futuro (requiere más GPU que la gratis): cambiar SadTalker por InfiniteTalk manteniendo la misma función `generar`, la app no necesita cambios.